# Rama 3 (punto C): entrenamiento en Google Colab

Fine-tuning de YOLO11m (COCO) sobre las imágenes fisheye de WoodScape, **sin rectificar**: 30 épocas, `imgsz=640`, `batch=16`, los mismos parámetros que la corrida de la Mac. La rama 4 usa los mismos valores salvo el batch físico (2) y la semilla. Usa el mismo código del repo (`fisheye_aug.py` y `tools/train_rama3.py`). Acá solo se entrena; la evaluación se hace en la Mac con `notebooks/rama3_finetune.ipynb`.

**Qué tiene que estar en Drive**, en una misma carpeta (por defecto `MyDrive/vpc2_rama3/`):
* `rama3_paquete.zip`: el paquete del repo (código, `config/`, manifest, listas y etiquetas). Está en `outputs/rama3/paquete/` del proyecto.
* `rgb_images.zip`: la carpeta `rgb_images` comprimida (8234 PNG). También sirve la carpeta sin comprimir, pero tarda mucho más en copiarse.

**Antes de correr:** *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → **GPU A100** (o L4 si no hay A100) y, si está disponible, *RAM alta*.

**Cómo correrlo:**
1. Con `PRUEBA = True`, *Ejecutar todo*. Monta Drive, descomprime, verifica las imágenes por sha256, entrena 1 época con el 5% del train y estima cuánto va a tardar la corrida completa.
2. Cambiar a `PRUEBA = False` y volver a ejecutar desde la celda de configuración. **Dejar la pestaña abierta** mientras entrena: si la sesión se desconecta, los checkpoints ya están en Drive y la última celda retoma desde la última época.
3. Al terminar, bajar de Drive la carpeta `runs_colab/rama3_<VARIANTE>` (`rama3_base` o `rama3_radial`) y copiarla en `outputs/rama3/colab/` del proyecto.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics==8.4.164

import glob
import hashlib
import os
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import torch
import ultralytics
from google.colab import drive

drive.mount("/content/drive")
print(f"ultralytics {ultralytics.__version__} | torch {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINGUNA'} | CPUs: {os.cpu_count()}")

### Configuración

Normalmente solo se toca `PRUEBA` (y `CARPETA_DRIVE` si los archivos están en otra carpeta). El resto está fijo para que coincida con la corrida de la Mac.

In [ ]:
CARPETA_DRIVE = Path("/content/drive/MyDrive/vpc2_rama3")  # carpeta de Drive con los dos zips
PAQUETE_ZIP = "rama3_paquete.zip"
IMAGENES_ZIP = "rgb_images.zip"  # o una carpeta rgb_images sin comprimir en CARPETA_DRIVE

PRUEBA = True  # True: chequeo corto (1 epoca, 5% del train). False: el entrenamiento real
VARIANTE = "base"  # punto C. "radial" entrena la extension con la augmentacion de distorsion radial
EPOCHS = 1 if PRUEBA else 30  # las mismas epocas que la corrida de la Mac y que la rama 4
FRACCION = 0.05 if PRUEBA else 1.0
CLOSE_MOSAIC = 0 if PRUEBA else 10  # en la prueba mantengo el mosaico para medir la velocidad real
BATCH = 16
IMGSZ = 640
WORKERS = min(8, os.cpu_count() or 2)
DEVICE = "0"

LOCAL = Path("/content/rama3")  # disco de la sesion: rapido, pero se borra al desconectar
SALIDA = CARPETA_DRIVE / ("runs_colab_prueba" if PRUEBA else "runs_colab")  # en Drive: sobrevive a una desconexion
LOCAL.mkdir(parents=True, exist_ok=True)
SALIDA.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "no hay GPU: Entorno de ejecucion -> Cambiar tipo de entorno -> GPU"
assert CARPETA_DRIVE.is_dir(), f"no existe {CARPETA_DRIVE}: corregir CARPETA_DRIVE"
print(f"prueba: {PRUEBA} | variante: {VARIANTE} | epocas: {EPOCHS} | fraccion: {FRACCION} | batch: {BATCH} | workers: {WORKERS}")
print(f"drive: {sorted(p.name for p in CARPETA_DRIVE.iterdir())}")
print(f"resultados -> {SALIDA}")

### Datos y código

Copio los zips de Drive al disco de la sesión y los descomprimo ahí: leer miles de imágenes directo desde Drive es muy lento. Excluyo la carpeta `__MACOSX` que agrega el compresor de la Mac. Si ya están descomprimidos de una ejecución anterior en la misma sesión, no repito el trabajo.

In [ ]:
def descomprimir(zip_drive, destino):
    # copia el zip al disco local y lo descomprime (sin la basura de macOS); devuelve el tiempo en segundos
    t0 = time.time()
    local_zip = LOCAL / zip_drive.name
    if not local_zip.exists():
        shutil.copy(zip_drive, local_zip)
    destino.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", str(local_zip), "-d", str(destino), "-x", "__MACOSX/*", "*.DS_Store"], check=True)
    local_zip.unlink()  # libero disco
    return time.time() - t0


def buscar_archivo(nombre, raiz):
    for d, _, archivos in os.walk(raiz):
        if nombre in archivos and "__MACOSX" not in d:
            return Path(d) / nombre
    return None


def buscar_imagenes(raiz):
    # la carpeta rgb_images que tiene las png adentro
    for d, subdirs, _ in os.walk(raiz):
        if "__MACOSX" in d:
            continue
        if "rgb_images" in subdirs and glob.glob(os.path.join(d, "rgb_images", "*.png")):
            return Path(d) / "rgb_images"
    return None


# paquete del repo
if buscar_archivo("fisheye_aug.py", LOCAL / "paquete") is None:
    print(f"paquete: {descomprimir(CARPETA_DRIVE / PAQUETE_ZIP, LOCAL / 'paquete'):.0f} s")
REPO = buscar_archivo("fisheye_aug.py", LOCAL / "paquete").parent

# imagenes: zip o carpeta sin comprimir
IMAGENES = buscar_imagenes(LOCAL / "dataset")
if IMAGENES is None:
    if (CARPETA_DRIVE / IMAGENES_ZIP).is_file():
        print(f"imagenes: {descomprimir(CARPETA_DRIVE / IMAGENES_ZIP, LOCAL / 'dataset') / 60:.1f} min")
    else:
        assert (CARPETA_DRIVE / "rgb_images").is_dir(), f"no encuentro {IMAGENES_ZIP} ni rgb_images/ en {CARPETA_DRIVE}"
        t0 = time.time()
        shutil.copytree(CARPETA_DRIVE / "rgb_images", LOCAL / "dataset/rgb_images", dirs_exist_ok=True)
        print(f"imagenes copiadas desde la carpeta: {(time.time() - t0) / 60:.1f} min")
    IMAGENES = buscar_imagenes(LOCAL / "dataset")

n_png = len(list(IMAGENES.glob("*.png")))
n_lbl = len(list((REPO / "yolo_ds/labels").glob("*.txt")))
print(f"repo: {REPO} | imagenes: {IMAGENES} ({n_png} png) | etiquetas: {n_lbl}")
assert n_png == 8234 and n_lbl == 8233, (n_png, n_lbl)
!df -h /content | tail -1

### Verificación de las imágenes

Regla del equipo: entrenar solo con archivos idénticos a los del manifest. Comparo el sha256 de las 7233 imágenes de train y val contra `manifest.csv`. Lo hago en la prueba; en la corrida real ya está verificado.

In [ ]:
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


if PRUEBA:
    man = pd.read_csv(REPO / "splits/manifest.csv")
    pool = man[man["split"] == "train_pool"]
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex:
        hashes = list(ex.map(sha256, [IMAGENES / f for f in pool["filename"]]))
    distintas = (pd.Series(hashes, index=pool.index) != pool["sha256"]).sum()
    print(f"train_pool: {len(pool)} imagenes | distintas al manifest: {distintas} | {time.time() - t0:.0f} s")
    assert distintas == 0, "hay imagenes distintas al manifest: no entrenar con esta copia"
else:
    print("verificacion hecha en la prueba")

### Entrenamiento

Llama a `tools/train_rama3.py`: arma `images/` + `labels/` con enlaces a las imágenes (sin copiarlas), controla los números del equipo (6148 / 1085 / 1000 imágenes y 74.909 / 13.176 / 11.910 cajas) y entrena. Con `VARIANTE = "base"` usa el trainer de Ultralytics sin cambios. Los pesos (`best.pt` = mejor mAP@50-95 en val, y `last.pt`), el `results.csv` y el log se escriben directamente en Drive.

In [ ]:
LOG = SALIDA / f"train_{VARIANTE}.log"
cmd = [
    sys.executable, str(REPO / "tools/train_rama3.py"), "--variante", VARIANTE, "--epochs", str(EPOCHS),
    "--device", DEVICE, "--imagenes", str(IMAGENES), "--etiquetas", str(REPO / "yolo_ds/labels"),
    "--listas", str(REPO / "yolo_ds"), "--ds", str(LOCAL / f"ds_{VARIANTE}"), "--salida", str(SALIDA),
    "--modelo", "yolo11m.pt", "--imgsz", str(IMGSZ), "--batch", str(BATCH), "--workers", str(WORKERS),
    "--fraccion", str(FRACCION), "--close-mosaic", str(CLOSE_MOSAIC), "--cache", "ram",
]
print(" ".join(cmd), flush=True)

t0 = time.time()
with open(LOG, "w") as log:
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        print(linea, end="", flush=True)
        log.write(linea)
        log.flush()
    p.wait()
DURACION = time.time() - t0
print(f"\ncodigo de salida {p.returncode} | {DURACION / 60:.1f} min")
assert p.returncode == 0, f"el entrenamiento fallo, ver {LOG}"

### Resumen

In [ ]:
RUN = SALIDA / f"rama3_{VARIANTE}"
df = pd.read_csv(RUN / "results.csv")
df.columns = df.columns.str.strip()
mejor = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
print(f"epocas: {len(df)} | mejor epoca {int(mejor['epoch'])}: mAP50-95 {mejor['metrics/mAP50-95(B)']:.4f} | mAP50 {mejor['metrics/mAP50(B)']:.4f} (val, ultralytics)")
print(f"best.pt: {(RUN / 'weights/best.pt').is_file()} | last.pt: {(RUN / 'weights/last.pt').is_file()}")

if PRUEBA:
    # estimacion de la corrida completa a partir de las barras de progreso de la prueba
    texto = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", LOG.read_text()).replace("\r", "\n")

    def velocidad(m):
        v = float(m.group(1))
        return v if m.group(2) == "it/s" else 1 / v

    tren = [velocidad(m) for m in re.finditer(r"640: .*?\d+/\d+ ([\d.]+)(it/s|s/it)", texto)]
    val = [float(m.group(1)) for m in re.finditer(r"mAP50-95\): 100%.*?\d+/\d+ [\d.]+(?:it/s|s/it) ([\d.]+)s", texto)]
    if tren and val:
        seg_epoca = -(-6148 // BATCH) / tren[-1] + val[-1]
        print(f"velocidad: {tren[-1]:.2f} it/s | val: {val[-1]:.0f} s -> una epoca completa ~{seg_epoca / 60:.1f} min")
        print(f"30 epocas: ~{30 * seg_epoca / 3600:.1f} h (mas unos minutos de arranque)")
    else:
        print(f"no pude leer la velocidad del log; la prueba tardo {DURACION / 60:.1f} min en total")
else:
    print(f"entrenamiento completo en {DURACION / 3600:.2f} h -> {RUN}")
df.tail(5)

### Si la sesión se desconectó

Los checkpoints están en Drive. Para retomar: volver a ejecutar las celdas hasta *Verificación* (montan Drive y vuelven a descomprimir los datos en las mismas rutas) y después esta celda, que sigue desde la última época guardada en `last.pt`.

In [ ]:
RETOMAR = False  # poner True solo si la corrida se corto a la mitad

if RETOMAR:
    import json
    sys.path.insert(0, str(REPO))
    from fisheye_aug import AUG_RADIAL, ENV_RADIAL, preparar_yolo_ds  # noqa: E402
    preparar_yolo_ds(REPO / "yolo_ds", REPO / "yolo_ds/labels", IMAGENES, LOCAL / f"ds_{VARIANTE}")  # mismas rutas que antes
    if VARIANTE == "radial":  # la augmentacion viaja por variable de entorno, como en entrenar()
        os.environ[ENV_RADIAL] = json.dumps({**AUG_RADIAL, "calib": str(REPO / "config/calibration.json")})
    ultimo = SALIDA / f"rama3_{VARIANTE}" / "weights/last.pt"
    print(f"retomando desde {ultimo}")
    trainer = "from fisheye_aug import TrainerFisheye as T" if VARIANTE == "radial" else "T = None"
    codigo = f"import sys; sys.path.insert(0, {str(REPO)!r}); from ultralytics import YOLO; {trainer}; YOLO({str(ultimo)!r}).train(resume=True, trainer=T)"
    cmd = [sys.executable, "-c", codigo]
    with open(SALIDA / f"train_{VARIANTE}_retomado.log", "w") as log:
        p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for linea in p.stdout:
            print(linea, end="", flush=True)
            log.write(linea)
        p.wait()
    print(f"codigo de salida {p.returncode}")